# 07 — Experimentos de fuga y explicabilidad: Destination_Port + SHAP

**Proyecto:** VIGÍA (Sentry) — TFC
**Objetivo:** cuantificar el impacto real de `Destination_Port` en el modelo por familias, comparando el rendimiento con y sin la feature. Cierra el experimento de fuga iniciado en Fase 1.

**Entradas:**
- `cic-ids2017-splits/train.parquet`, `val.parquet`
- `src/models/familias_sin_puerto.json` (modelo ya entrenado)

**Hipótesis a verificar:**
Si `Destination_Port` es un atajo, el modelo **con** puerto mejorará notablemente en las clases con puerto exclusivo (Bot, Brute-force). Si el F1 macro se mantiene igual, el puerto no aporta señal útil.

**Motivo de seguridad:** demostrar empíricamente si una feature es un atajo permite decidir con criterio si incluirla en producción y cómo documentar su limitación.

In [ ]:
# ============================================
# Setup notebook 07
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("VIGIA_Experimentos") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version}")

## 1. Carga de splits y definición de features

Cargamos train y val, y definimos dos listas de features:
- `feature_cols_con_puerto`: incluye `Destination_Port`
- `feature_cols_sin_puerto`: la misma lista sin esa columna

**Importante:** mantendremos el resto de decisiones idénticas al notebook 06 (mismo muestreo, mismos hiperparámetros, misma semilla). Solo cambia la lista de features.

In [ ]:
# ============================================
# Carga de splits y definición de features
# ============================================
from pyspark.sql.types import NumericType

train_df = spark.read.parquet(f"{BASE_SPLITS}/train.parquet")
val_df   = spark.read.parquet(f"{BASE_SPLITS}/val.parquet")

# Features CON puerto
excluir = {"Label", "target_bin", "target_family"}
feature_cols_con_puerto = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]

# Features SIN puerto
feature_cols_sin_puerto = [c for c in feature_cols_con_puerto if c != "Destination_Port"]

print(f"Features con puerto: {len(feature_cols_con_puerto)}")
print(f"Features sin puerto: {len(feature_cols_sin_puerto)}")
print(f"Diferencia: {set(feature_cols_con_puerto) - set(feature_cols_sin_puerto)}")

print(f"\nTrain: {train_df.count():,}")
print(f"Val:   {val_df.count():,}")

## 2. Preparación del target y muestreo

Reaplicamos el mapeo de familias (0-6) y el muestreo estratificado al 50% para mantener las mismas condiciones que en el 06.

In [ ]:
# ============================================
# Mapeo de familias + muestreo
# ============================================
from pyspark.sql.functions import col, when
from functools import reduce
from pyspark.sql import DataFrame

familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

def aplicar_mapeo(df):
    expr = None
    for fam, idx in mapeo.items():
        cond = when(col("target_family") == fam, idx)
        expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
    return df.withColumn("target_family_idx", expr.otherwise(-1))

train_df = aplicar_mapeo(train_df)
val_df   = aplicar_mapeo(val_df)

# Muestreo al 50%
print("Muestreando train al 50%...")
parts = [train_df.filter(col("target_family") == fam).sample(fraction=0.5, seed=42)
         for fam in familias]
train_sample = reduce(DataFrame.unionByName, parts)
train_sample.cache()

print(f"Train muestreado: {train_sample.count():,}")

In [ ]:
# ============================================
# Conversión a numpy — features CON puerto
# ============================================
import numpy as np
import time, gc

print("Convirtiendo train (con puerto)...")
t0 = time.time()
train_pd = train_sample.select(feature_cols_con_puerto + ["target_family_idx"]).toPandas()
X_train = train_pd[feature_cols_con_puerto].values.astype(np.float32)
y_train = train_pd["target_family_idx"].values.astype(np.int8)
del train_pd; gc.collect()
print(f"✅ X_train: {X_train.shape}  ({X_train.nbytes/1e6:.0f} MB) en {time.time()-t0:.1f}s")

print("\nConvirtiendo val...")
t0 = time.time()
val_pd = val_df.select(feature_cols_con_puerto + ["target_family_idx"]).toPandas()
X_val = val_pd[feature_cols_con_puerto].values.astype(np.float32)
y_val = val_pd["target_family_idx"].values.astype(np.int8)
del val_pd; gc.collect()
print(f"✅ X_val: {X_val.shape} en {time.time()-t0:.1f}s")

In [ ]:
# ============================================
# Pesos por clase (idénticos al 06)
# ============================================
from sklearn.utils.class_weight import compute_sample_weight

pesos = compute_sample_weight(class_weight="balanced", y=y_train)
print(f"Pesos: min={pesos.min():.2f}, max={pesos.max():.2f}")

## 3. Entrenamiento con Destination_Port

Mismo XGBoost, mismos hiperparámetros, misma semilla. **Única diferencia:** se incluye `Destination_Port` en la matriz de features.

In [ ]:
# ============================================
# Entrenar modelo por familias CON puerto
# ============================================
import xgboost as xgb
import time

modelo_con_puerto = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=7,
    eval_metric="mlogloss",
    max_depth=6,
    n_estimators=200,
    learning_rate=0.1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print("Entrenando modelo con Destination_Port (3-8 min)...")
t0 = time.time()
modelo_con_puerto.fit(
    X_train, y_train,
    sample_weight=pesos,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f"✅ Entrenado en {time.time()-t0:.1f}s")

## 4. Evaluación y comparativa por clase

Comparamos:
- F1 macro global (con vs sin)
- F1 por clase (con vs sin)
- Deltas (cuánto cambia cada clase al añadir el puerto)

**Lo que buscamos:** si las clases con puerto exclusivo (Bot, Brute-force) mejoran mucho mientras el resto se mantiene, el puerto es un atajo.

## 6. Explicabilidad con SHAP

**Objetivo:** entender qué features usa el modelo para cada familia.

**Motivo de seguridad:** un analista de SOC necesita saber *por qué* el modelo marca un flujo como ataque. SHAP da esa explicación de forma matemáticamente sólida.

**Enfoque:** calculamos dos vistas:
1. **Global:** qué features pesan más en todo el modelo (todas las clases agregadas).
2. **Por clase:** qué features definen cada familia concreta (útil para identificar si el modelo aprende comportamiento real o atajos).

In [ ]:
# ============================================
# Reconstrucción completa del estado del 07
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import NumericType
from functools import reduce
from pyspark.sql import DataFrame
import numpy as np
import gc

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("VIGIA_07_rebuild") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"✅ Spark {spark.version}")

# Cargar splits
train_df = spark.read.parquet(f"{BASE_SPLITS}/train.parquet")
val_df   = spark.read.parquet(f"{BASE_SPLITS}/val.parquet")

# Features
excluir = {"Label", "target_bin", "target_family"}
feature_cols_con_puerto = [
    c for c in train_df.columns
    if c not in excluir and isinstance(train_df.schema[c].dataType, NumericType)
]
feature_cols_sin_puerto = [c for c in feature_cols_con_puerto if c != "Destination_Port"]

# Mapeo familias
familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

def aplicar_mapeo(df):
    expr = None
    for fam, idx in mapeo.items():
        cond = when(col("target_family") == fam, idx)
        expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
    return df.withColumn("target_family_idx", expr.otherwise(-1))

val_df = aplicar_mapeo(val_df)

# Cargar val SIN puerto (lo que necesita SHAP)
print("Convirtiendo val sin puerto...")
val_pd = val_df.select(feature_cols_sin_puerto + ["target_family_idx"]).toPandas()
X_val_sin = val_pd[feature_cols_sin_puerto].values.astype(np.float32)
y_val_sin = val_pd["target_family_idx"].values.astype(np.int8)
del val_pd; gc.collect()

print(f"✅ X_val_sin: {X_val_sin.shape} | y_val_sin: {y_val_sin.shape}")

# Cargar el modelo guardado
import xgboost as xgb
modelo_sin_puerto = xgb.XGBClassifier()
modelo_sin_puerto.load_model("/content/sentry/src/models/familias_sin_puerto.json")
print("✅ Modelo sin puerto cargado")

# Verificar
print(f"\nClases en y_val_sin: {sorted(np.unique(y_val_sin))}")
print(f"Familias esperadas: {list(range(7))}")

In [ ]:
# ============================================
# Preparar muestra pequeña para SHAP
# ============================================
!pip install shap -q

import shap
import numpy as np

np.random.seed(42)
N_MUESTRA = 2000
idx = np.random.choice(len(X_val_sin), N_MUESTRA, replace=False)

X_shap = X_val_sin[idx]
y_shap = y_val_sin[idx]

print(f"Muestra para SHAP: {X_shap.shape}")
print(f"Distribución de clases en la muestra:")
for i, fam in enumerate(familias):
    n = (y_shap == i).sum()
    if n > 0:
        print(f"  {fam:15s} {n:>5}")

In [ ]:
# ============================================
# Calcular valores SHAP
# ============================================
import time

print("Calculando SHAP (puede tardar 2-5 min)...")
t0 = time.time()

explainer = shap.TreeExplainer(modelo_sin_puerto)
shap_values = explainer.shap_values(X_shap)

# Normalizar a array 3D (n_muestras, n_features, n_clases)
if isinstance(shap_values, list):
    shap_values_array = np.stack(shap_values, axis=-1)
else:
    shap_values_array = shap_values

print(f"✅ Calculado en {time.time()-t0:.1f}s")
print(f"Shape SHAP: {shap_values_array.shape}")

In [ ]:
# ============================================
# Importancia global (media de |SHAP| sobre todas las clases)
# ============================================
import matplotlib.pyplot as plt

importancia_global = np.abs(shap_values_array).mean(axis=(0, 2))

top_idx = np.argsort(importancia_global)[::-1][:20]
top_features = [feature_cols_sin_puerto[i] for i in top_idx]
top_values = importancia_global[top_idx]

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_features)), top_values[::-1], color="steelblue")
plt.yticks(range(len(top_features)), top_features[::-1])
plt.xlabel("Importancia SHAP media (|valor|)")
plt.title("Top 20 features más importantes (todas las clases)")
plt.tight_layout()
plt.savefig("/content/sentry/docs/shap_importancia_global.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================
# Top 10 features por clase
# ============================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, fam in enumerate(familias):
    import_clase = np.abs(shap_values_array[:, :, i]).mean(axis=0)
    top = np.argsort(import_clase)[::-1][:10]
    feats = [feature_cols_sin_puerto[j] for j in top]
    vals = import_clase[top]

    ax = axes[i]
    ax.barh(range(10), vals[::-1], color="coral")
    ax.set_yticks(range(10))
    ax.set_yticklabels(feats[::-1], fontsize=8)
    ax.set_title(f"{fam}", fontsize=11)
    ax.set_xlabel("|SHAP| medio", fontsize=8)

axes[-1].axis("off")
axes[-1].text(0.1, 0.5,
    "8 clases\n1 gráfico por familia\n\nLas barras muestran\nlas 10 features con\nmayor contribución\na cada clase.",
    fontsize=10, va="center")

plt.tight_layout()
plt.savefig("/content/sentry/docs/shap_por_clase.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================
# Guardar valores SHAP, diario y commit + push
# ============================================
import json

top_features_por_clase = {}
for i, fam in enumerate(familias):
    import_clase = np.abs(shap_values_array[:, :, i]).mean(axis=0)
    top = np.argsort(import_clase)[::-1][:10]
    top_features_por_clase[fam] = [
        {"feature": feature_cols_sin_puerto[j],
         "importancia": float(import_clase[j])}
        for j in top
    ]

with open("/content/sentry/src/models/shap_top_features_por_clase.json", "w") as f:
    json.dump(top_features_por_clase, f, indent=2)

top_global = [
    {"feature": feature_cols_sin_puerto[j],
     "importancia": float(importancia_global[j])}
    for j in top_idx
]

%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Explicabilidad SHAP del modelo por familias

**Qué probé:**
- SHAP TreeExplainer sobre el modelo por familias sin puerto.
- Cálculo sobre muestra aleatoria de 2000 flujos de validación.
- Importancia global y por clase.

**Top 5 features globales:**
"""
for i, item in enumerate(top_global[:5], 1):
    entrada += f"{i}. {item['feature']}: {item['importancia']:.4f}\n"

entrada += """
**Artefactos generados:**
- `docs/shap_importancia_global.png`
- `docs/shap_por_clase.png`
- `src/models/shap_top_features_por_clase.json`

**Próximos pasos:**
- Dashboard con explicaciones SHAP en cada alerta.
- Evasión adversarial controlada (Fase 3).
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

# Commit
!git add .
!git commit -m "SHAP: importancia global y por clase del modelo de familias"

# Push con token
from getpass import getpass
TOKEN = getpass("token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push

## 10. Evaluación final sobre test

Se evalúan **una sola vez** los dos modelos de familias (con y sin `Destination_Port`) sobre `test.parquet`, que no se ha tocado hasta ahora.

**Motivo de seguridad:** la misma regla que en el binario. Test se usa una vez y no se reajusta nada tras ver los resultados.

**Incluye:** análisis específico sobre la familia "Otros" (Infiltration + Heartbleed), que se reservó íntegramente para test.

In [ ]:
# ============================================
# Reconstruir estado mínimo
# ============================================
import os
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.sql.types import NumericType
import numpy as np
import xgboost as xgb
import gc

GITHUB_USER = "AguCS231"
REPO_NAME   = "sentry"
REPO_LOCAL  = f"/content/{REPO_NAME}"
BASE_SPLITS = "/content/drive/MyDrive/cic-ids2017-splits"

if not os.path.exists(REPO_LOCAL):
    %cd /content
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    %cd {REPO_LOCAL}
    !git config --global user.email "srcasares2@gmail.com"
    !git config --global user.name "{GITHUB_USER}"
else:
    %cd {REPO_LOCAL}

try:
    drive.mount('/content/drive')
except:
    pass

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .appName("VIGIA_07_cierre") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()

print(f"✅ Spark {spark.version}")

test_df = spark.read.parquet(f"{BASE_SPLITS}/test.parquet")
print(f"Test: {test_df.count():,} filas")

# Features
excluir = {"Label", "target_bin", "target_family"}
feature_cols_con_puerto = [
    c for c in test_df.columns
    if c not in excluir and isinstance(test_df.schema[c].dataType, NumericType)
]
feature_cols_sin_puerto = [c for c in feature_cols_con_puerto if c != "Destination_Port"]

# Mapeo
familias = ['BENIGN', 'DoS', 'Probe', 'DDoS', 'Brute-force', 'Web', 'Bot']
mapeo = {fam: i for i, fam in enumerate(familias)}

expr = None
for fam, idx in mapeo.items():
    cond = when(col("target_family") == fam, idx)
    expr = cond if expr is None else expr.when(col("target_family") == fam, idx)
test_df = test_df.withColumn("target_family_idx", expr.otherwise(-1))

# Cargar SOLO el modelo sin puerto (el baseline elegido)
modelo_sin = xgb.XGBClassifier()
modelo_sin.load_model(f"{REPO_LOCAL}/src/models/familias_sin_puerto.json")
print("✅ Modelo sin puerto cargado")

In [ ]:
# ============================================
# Evaluación en test del modelo sin puerto (baseline)
# ============================================
from sklearn.metrics import f1_score, classification_report

# Test sin "Otros"
test_sin_otros = test_df.filter("target_family_idx >= 0")
print(f"Test para evaluación principal: {test_sin_otros.count():,} filas")

# Convertir a numpy
print("\nConvirtiendo test sin puerto...")
test_pd = test_sin_otros.select(feature_cols_sin_puerto + ["target_family_idx"]).toPandas()
X_test = test_pd[feature_cols_sin_puerto].values.astype(np.float32)
y_test = test_pd["target_family_idx"].values.astype(np.int8)
del test_pd; gc.collect()
print(f"✅ X_test: {X_test.shape}")

# Predicciones
y_pred_test = modelo_sin.predict(X_test)
f1_macro_test = f1_score(y_test, y_pred_test, average="macro")
f1_weighted_test = f1_score(y_test, y_pred_test, average="weighted")

print("\n=== Modelo SIN puerto — TEST ===\n")
print(f"F1 macro:    {f1_macro_test:.4f}")
print(f"F1 weighted: {f1_weighted_test:.4f}")

print("\nF1 por clase:")
print(classification_report(
    y_test, y_pred_test,
    labels=list(range(7)),
    target_names=familias,
    digits=4,
    zero_division=0,
))

# Guardar variable para el diario
f1_por_clase_test = {
    fam: f1_score(y_test, y_pred_test, labels=[i], average="macro", zero_division=0)
    for i, fam in enumerate(familias)
}

## 11. Análisis sobre la familia "Otros"

`Otros` (Infiltration + Heartbleed, 47 muestras) se reservó **íntegramente para test** y nunca apareció en entrenamiento.

**Este es el experimento de generalización:** ¿detecta el modelo ataques que nunca ha visto?

**Motivo de seguridad:** en producción, los ataques nuevos no están en el train. Un detector que solo memoriza lo que ha visto es trivialmente evadible con una variante nueva.

In [ ]:
# ============================================
# Comportamiento sobre "Otros" (no vistos en train)
# ============================================
otros_df = test_df.filter("target_family = 'Otros'")
print(f"Muestras en Otros: {otros_df.count()}")

otros_pd = otros_df.select(feature_cols_sin_puerto + ["Label"]).toPandas()
X_otros = otros_pd[feature_cols_sin_puerto].values.astype(np.float32)

# El modelo predice una de las 7 familias. Cualquier predicción != BENIGN se considera "detectado"
pred_otros = modelo_sin.predict(X_otros)
otros_pd["prediccion_idx"] = pred_otros
otros_pd["prediccion_familia"] = [familias[p] for p in pred_otros]
otros_pd["detectado"] = pred_otros != 0  # != BENIGN

print("\n=== Detalle por muestra ===")
print(otros_pd[["Label", "prediccion_familia", "detectado"]].to_string())

# Resumen
total_otros = len(otros_pd)
detectados = otros_pd["detectado"].sum()
print(f"\n=== Resumen ===")
print(f"Detectados como ataque: {detectados} / {total_otros} ({detectados/total_otros*100:.1f}%)")
print(f"Clasificados como BENIGN: {total_otros - detectados} ({(total_otros-detectados)/total_otros*100:.1f}%)")

## 12. Decisión formal y cierre

Tras el análisis completo, se documentan las decisiones finales del notebook 07.

In [ ]:
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

total_otros = len(otros_pd)
detectados = otros_pd["detectado"].sum()
recall_otros = detectados / total_otros * 100

entrada = f"""

---

## {fecha} — Cierre del notebook 07: evaluación en test y decisión formal

**Evaluación en TEST (modelo sin puerto, baseline elegido):**
- F1 macro: {f1_macro_test:.4f}
- F1 weighted: {f1_weighted_test:.4f}
- F1 por clase:
"""
for fam in familias:
    entrada += f"  - {fam}: {f1_por_clase_test[fam]:.4f}\n"

entrada += f"""
**Comparativa con/sin puerto:** realizada en validación (notebook 07, sección 5). No se repitió en test para preservar la regla de uso único del conjunto de test.

**Experimento de generalización (familia "Otros", no vista en train):**
- Total muestras: {total_otros}
- Detectados como ataque: {detectados} ({recall_otros:.1f}%)
- Clasificados como BENIGN: {total_otros - detectados} ({100-recall_otros:.1f}%)

**Hallazgos SHAP documentados:**
- Las features dominantes son de comportamiento real del flujo (tamaños, flags TCP, timing), no identificadores.
- `Init_Win_bytes_forward` y `Init_Win_bytes_backward` dominan la mayoría de clases. Estas features dependen del sistema operativo del emisor y son manipulables por un atacante. Se documentan como limitación conocida y se plantea experimento adicional sin ellas (notebook 08).

**Decisión formal:**
- Modelo de familias oficial del proyecto: **sin Destination_Port**.
- Coherencia con el binario y con el análisis de fuga.

**Limitaciones conocidas:**
- Train al 50% por RAM.
- Clase Bot con precision baja (0.59) por peso desproporcionado.
- Modelo con puerto no persistido en disco (solo análisis en validación).

**Próximos pasos (notebook 08):**
- Experimento sin `Init_Win_bytes`.
- Evasión adversarial controlada.
- Dashboard con SHAP integrado.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Cierre notebook 07: evaluación en test y decisión formal"

from getpass import getpass
TOKEN = getpass("Pega tu token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push

In [ ]:
%cd /content/sentry

from datetime import datetime
fecha = datetime.now().strftime("%Y-%m-%d")

entrada = f"""

---

## {fecha} — Interpretación del resultado sobre "Otros"

**Datos:** 2 de 47 muestras detectadas (4.3%). Infiltration y Heartbleed se clasifican mayoritariamente como BENIGN.

**Análisis:**
El modelo alcanza F1 macro 0.96 sobre las 7 clases que ha visto en entrenamiento, pero falla en las 2 clases que se reservaron íntegramente para test. Esto no invalida el modelo, pero sí acota su alcance:

1. El modelo detecta con precisión las familias conocidas.
2. El modelo NO generaliza a ataques con patrones muy distintos a los vistos.
3. Coincide con el comportamiento esperado en producción: los ataques nuevos suelen colarse.

**Implicación de seguridad:**
Un IDS entrenado con un dataset cerrado no es suficiente contra amenazas emergentes. Necesita, además:
- Reentrenamiento periódico con tráfico reciente.
- Detección no supervisada complementaria para anomalías.
- Alertas humanas sobre flujos "extraños" aunque el modelo los clasifique como BENIGN con confianza media.

**Decisión:**
- Este hallazgo se documenta como **limitación conocida** del sistema.
- Se plantea como **línea de trabajo futuro** la inclusión de un módulo de detección de anomalías no supervisado (autoencoder o Isolation Forest) para complementar al clasificador supervisado.
- Se convierte en argumento a favor de la sección de conclusiones: el TFC no vende un detector perfecto, vende un detector honesto con límites explícitos.
"""

with open("docs/diario.md", "a") as f:
    f.write(entrada)

print("✅ Diario actualizado")

!git add .
!git commit -m "Interpretación del resultado sobre ataques no vistos (Otros)"

from getpass import getpass
TOKEN = getpass("Pega tu token: ")
!git remote set-url origin https://{TOKEN}@github.com/AguCS231/sentry.git
!git push